# QMS Wait-Time ML Pipeline

**Purpose**: Train and validate a HistGradientBoosting model to predict queue waiting time for Malaysian bank branches.

**Pipeline Lifecycle**:
1. Data Ingestion (synthetic + optional external datasets)
2. Data Cleansing (abandoned tickets, outliers, dedup)
3. Feature Engineering (28 features aligned with production)
4. Model Training (temporal split, sample weighting, quantile p75 loss)
5. Evaluation & Comparison (ML vs Formula baseline)

**System**: QMS (Queue Management System) — online booking + walk-in, multi-counter, CreatedAt FIFO call ordering

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

print('Libraries loaded.')

## 1. Data Ingestion

Generate synthetic data that matches production system behavior (the simulation cells below are copied verbatim from `ml/generate_seed_data.py`; edit both together):
- **Production services**: codes, names, ticket prefixes and default minutes from `database/insert-service-types-all-branches.sql` + `insert-service-queues-all-branches.sql` (1 service = 1 queue = 1 prefix: A = Account Opening, D = Deposit, W = Withdraw, L = Loan, C = Credit Card, F = Fixed Deposit, I = Investment, B = Bancassurance, T = ATM/CDM, O = Online Banking, S = Business Banking, R = Remittance, Q = Customer Service)
- **Randomized branch-days**: random real BranchCode (101001..101251), 1..8 Active counters (the seed creates 8 Closed counters per branch; 1-2 counters are common), each counter with a realistic grouping, a random subset, a single service or all services; counters may open late, close early or take a break; load from very quiet to over capacity; opening hours mostly 09:30-16:00 with some other windows; 60-min slots aligned to opening time; `OnlineSlotsPerSlot` drawn per branch-day per service from 2..8 (bookings beyond capacity become walk-ins)
- **CreatedAt FIFO** call ordering (matches `QueueCallNextSelector`), teller reaction time, walk-ins that left still get called (then Missed)
- **People Ahead (per counter)**: for each Active counter that serves my lane, `PeopleAheadCallNext` counts eligible, not-yet-called tickets in THAT counter's lanes that Call Next picks before me by (CreatedAt, Id), and takes the minimum over those counters (same-lane only if no counter serves my lane); `CrossLaneQueueLength` = waiting count over the union S of those lanes. Same definition as live `CallNextQueueMath`
- **Counter state**: `IdleListeningCounters` (listening counters with no Called/Serving ticket) and `CounterSimEtaMinutes` (deterministic Call Next drain = live `CounterSimulationEstimator`)
- **RollingAvgServiceMinutes** = live `RollingServiceAverage` rule: mean of the last 30 completed services (ignoring < 30 s); fewer than 5 valid -> `DefaultAvgServiceMinutes`. Some branch-days start with no history (or only seconds-long test services) so the fallback is exercised
- **EnqueueSequence** = per branch-day rank by CreatedAt across all services (same as live)
- **Check-in time** for online bookings: 20% very early (before the early-call window), 55% early, 25% late; `QueueEligibleAt = max(CheckInAt, SlotStart - EarlyCall)`
- **Target** = `CalledAt - snapshot`: one snapshot at `QueueEligibleAt` per called ticket (= `MlTrainingObservation.ActualWaitingMinutes`) plus, for 35% of tickets, one extra snapshot at a random moment of the wait (also before the early-call window), because the live ETA is predicted at any time while the customer waits

In [ ]:
# ── Branch & Service configs (matching the production seed scripts in database/) ──

# Real production BranchCodes: insert-branches-from-paste.sql gives 251 branches, BranchCode = 101000 + row.
# Each simulated branch-day draws a random code, so BranchCode carries no simulated signal (live branches
# the simulation never named would otherwise get arbitrary tree splits).
BRANCH_CODE_MIN = 101001
BRANCH_CODE_MAX = 101251

# Branch service window (BRANCH_OPERATING_HOURS seed): Mon-Fri 09:30-16:00. Managers can change the hours
# (Schedule & capacity), so some branch-days use other windows. Slots are aligned to opening time
# (QmsQueueService.AlignSlot anchor) and a booked slot must end by closing time.
OPEN_MIN = 9 * 60 + 30    # 570 = 09:30
CLOSE_MIN = 16 * 60       # 960 = 16:00
HOURS_CHOICES = [  # (open, close) in minutes from midnight, probability
    ((570, 960), 0.80),   # 09:30-16:00 (seed)
    ((570, 990), 0.06),   # 09:30-16:30
    ((540, 1020), 0.05),  # 09:00-17:00
    ((540, 960), 0.04),   # 09:00-16:00
    ((570, 1050), 0.03),  # 09:30-17:30
    ((600, 1080), 0.02),  # 10:00-18:00
]
DRAIN_AFTER_CLOSE_MIN = 90  # open counters keep calling the remaining queue for up to 90 min after closing
SLOT_DURATION = 60          # Branch.SlotDurationMinutes

# Production services: code, name, prefix (SERVICE_QUEUES.TicketPrefix) and DefaultAvgServiceMinutes.
# "weight" = share of arrivals (sim only, jittered per branch-day).
SERVICES = [
    {"code": "ACC_OPEN", "name": "Account Opening",           "prefix": "A", "avg_min": 15, "weight": 0.08},
    {"code": "DEPOSIT",  "name": "Deposit Services",          "prefix": "D", "avg_min": 3,  "weight": 0.16},
    {"code": "WITHDRAW", "name": "Withdrawal Services",       "prefix": "W", "avg_min": 4,  "weight": 0.06},
    {"code": "LOAN",     "name": "Loan / Financing",          "prefix": "L", "avg_min": 30, "weight": 0.08},
    {"code": "CCARD",    "name": "Credit Card Services",      "prefix": "C", "avg_min": 10, "weight": 0.10},
    {"code": "FDEP",     "name": "Fixed Deposit",             "prefix": "F", "avg_min": 15, "weight": 0.06},
    {"code": "INVUT",    "name": "Investment / Unit Trust",   "prefix": "I", "avg_min": 30, "weight": 0.04},
    {"code": "BANCA",    "name": "Insurance / Bancassurance", "prefix": "B", "avg_min": 25, "weight": 0.06},
    {"code": "ATMCDM",   "name": "ATM / CDM Services",        "prefix": "T", "avg_min": 5,  "weight": 0.10},
    {"code": "ONLBANK",  "name": "Online Banking Support",    "prefix": "O", "avg_min": 5,  "weight": 0.05},
    {"code": "BIZBANK",  "name": "Business Banking",          "prefix": "S", "avg_min": 20, "weight": 0.06},
    {"code": "REMIT",    "name": "Remittance / Transfer",     "prefix": "R", "avg_min": 15, "weight": 0.05},
    {"code": "CUSTSVC",  "name": "Queue / Customer Service",  "prefix": "Q", "avg_min": 10, "weight": 0.14},
]
ALL_PREFIXES = [s["prefix"] for s in SERVICES]

# Counters: the seed creates 8 Closed counters per branch with no allowed services; the manager opens some
# and ticks their services (COUNTER_ALLOWED_SERVICES). The real groupings are unknown, so each counter gets
# one of: a realistic teller grouping, a random subset, a single service, or all services.
MAX_COUNTERS = 8
N_COUNTER_WEIGHTS = [0.16, 0.16, 0.15, 0.14, 0.12, 0.10, 0.09, 0.08]  # P(1..8 Active counters)
PROFILES = [
    ["D", "W", "Q"],                 # cash teller: deposit, withdraw, customer service
    ["D", "W", "T"],                 # cash teller: deposit, withdraw, ATM/CDM
    ["D", "W"],                      # cash only
    ["L", "C"],                      # credit: loan, credit card
    ["A", "F", "I", "B"],            # sales: account opening, fixed deposit, investment, bancassurance
    ["A", "B"],                      # account opening + bancassurance
    ["A", "F"],                      # account opening + fixed deposit
    ["S", "O", "R"],                 # business banking, online banking, remittance
    ["Q", "T", "O"],                 # customer service desk
    ["D", "W", "L", "C", "Q"],       # multi-skill
    ["A", "D", "W", "F", "Q", "R"],  # universal teller
]
COUNTER_MODE_WEIGHTS = {"profile": 0.40, "random": 0.30, "single": 0.15, "all": 0.15}

# Counter availability changes during the day (manager opens a counter late, closes it early, or a break).
P_COUNTER_LATE_OPEN = 0.20
P_COUNTER_EARLY_CLOSE = 0.12
P_COUNTER_BREAK = 0.25

# Load = arrival rate relative to the open counters' capacity (rho), from very quiet to over capacity.
LOAD_LEVELS = [  # (rho_low, rho_high, probability)
    (0.02, 0.15, 0.18),   # very quiet
    (0.15, 0.45, 0.24),   # quiet
    (0.45, 0.80, 0.30),   # normal
    (0.80, 1.05, 0.20),   # busy
    (1.05, 1.40, 0.08),   # very busy (queue builds up)
]

# Realistic manager settings for Branch.OnlineEarlyCallMinutes (drawn per branch-day)
EARLY_CALL_CHOICES = [0, 5, 10, 15, 20, 30]

# Manager setting ServiceType.OnlineSlotsPerSlot (max online bookings per slot for one service). Drawn per
# branch-day per service, uniform integer 2..8.
ONLINE_SLOTS_MIN = 2
ONLINE_SLOTS_MAX = 8

# RollingAvgServiceMinutes = RollingServiceAverage rule (C#): mean of the last 30 completed services of the
# service with completion <= snapshot, ignoring durations < 30 s; fewer than 5 valid -> DefaultAvgServiceMinutes.
ROLLING_WINDOW = 30
ROLLING_MIN_SAMPLES = 5
ROLLING_MIN_DURATION_MIN = 0.5
HISTORY_MODE_WEIGHTS = {"none": 0.30, "few": 0.20, "full": 0.50}  # SERVICE_LOGS before the day starts

# Teller reaction time between a counter becoming free (or a ticket becoming eligible) and Call Next.
CALL_LATENCY_MEAN_MIN = 0.3
# Walk-ins that left: still Waiting, get called, counter waits / marks Missed (no SERVICE_LOGS row).
P_WALKIN_LEFT = 0.04
# Display-time snapshots: probability of one extra snapshot at a random moment of the wait.
EXTRA_SNAPSHOT_PROB = 0.35

In [ ]:
from __future__ import annotations  # PEP 604 annotations on Python < 3.10
from dataclasses import dataclass, field

@dataclass
class Ticket:
    """A simulated queue ticket."""
    ticket_id: int
    branch_code: int
    service_code: str
    prefix: str
    default_avg_min: float
    online_slots_per_slot: int
    entry_type: int  # 0=OnlineBooked, 1=WalkIn
    checked_in: bool
    created_at: float  # minutes from midnight (online = booking time, may be hours earlier)
    slot_start: float | None = None  # online: booked slot; walk-in: assigned bucket (AssignedSlotStart)
    slot_end: float | None = None
    checkin_at: float | None = None  # physical check-in (online) / created_at (walk-in)
    queue_eligible_at: float | None = None  # walk-in = created_at, online = max(checkin_at, slot_start - early_call)
    left: bool = False  # walk-in left before being called (still Waiting, gets called then Missed)
    called_at: float | None = None  # CalledAt: a counter picks the ticket (target end point)
    service_end_at: float | None = None  # counter becomes free again
    counter_id: int | None = None
    enqueue_seq: int = 0  # per branch-day rank by (created_at, ticket_id), across all services


@dataclass
class Counter:
    """A simulated service counter (Active while inside one of its intervals)."""
    counter_id: int
    allowed_prefixes: list[str]
    intervals: list[tuple[float, float]] = field(default_factory=list)
    next_free: float = 0.0
    retired: bool = False

    def is_active(self, t: float) -> bool:
        return any(a <= t < b for a, b in self.intervals)


def hour_traffic(hour: int) -> float:
    return {8: 0.8, 9: 1.4, 10: 1.6, 11: 1.3, 12: 0.7, 13: 0.8, 14: 1.3, 15: 1.5, 16: 1.2, 17: 0.9}.get(hour, 0.5)


def day_traffic(dow: int) -> float:
    # 0=Mon..4=Fri, 5=Sat, 6=Sun (python weekday; live WaitTimeFeatureBuilder uses the same encoding)
    return {0: 1.25, 1: 1.0, 2: 0.95, 3: 0.95, 4: 1.15}.get(dow, 0.0)

In [ ]:
from __future__ import annotations  # PEP 604 annotations on Python < 3.10

def rolling_avg_service_minutes(history: list[tuple[float, float]], at: float, default_min: float) -> float:
    """RollingAvgServiceMinutes (same rule as C# RollingServiceAverage).

    history = (completion_time, duration_min) of completed services of this service (SERVICE_LOGS).
    Mean of the last ROLLING_WINDOW completions <= at whose duration >= 30 s; fewer than
    ROLLING_MIN_SAMPLES valid ones -> the service's DefaultAvgServiceMinutes.
    """
    valid = sorted((t, d) for t, d in history if t <= at and d >= ROLLING_MIN_DURATION_MIN)
    last = [d for _, d in valid[-ROLLING_WINDOW:]]
    if len(last) < ROLLING_MIN_SAMPLES:
        return float(default_min)
    return float(np.mean(last))


def is_waiting(o: Ticket, snap: float) -> bool:
    """Call Next eligible and not yet called at snap (live CallNextQueueMath.EligibleWaiting).

    Walk-in: created (checked in on creation). Online: checked in and SlotStart - EarlyCall <= snap,
    i.e. queue_eligible_at <= snap. Walk-ins that left are still Waiting until called.
    """
    return (o.checked_in and o.queue_eligible_at is not None and o.queue_eligible_at <= snap
            and (o.called_at is None or o.called_at > snap))


def heard_lanes(prefix: str, active: list[Counter]) -> tuple[list[set[str]], set[str]]:
    """Listening counters' prefix sets and the heard lane set S (live CallNextQueueMath.ResolveHeardLanesAsync).

    Only Active counters count. A counter listens to my prefix if its allowed_prefixes contain it.
    S = union of the listening counters' prefixes (includes mine); S = {my prefix} if no counter serves it.
    """
    listening = [set(c.allowed_prefixes) for c in active if prefix in c.allowed_prefixes]
    heard = set().union(*listening) if listening else {prefix}
    return listening, heard


def people_ahead_call_next(t: Ticket, waiting: list[Ticket], listening: list[set[str]]) -> int:
    """PeopleAheadCallNext (live CallNextQueueMath.CountPeopleAheadAsync).

    For each listening counter, the waiting tickets in THAT counter's prefixes picked before me by
    (created_at, ticket_id) (= live (CreatedAt, Id)); minimum over the counters. If no Active counter serves
    my prefix, same-prefix tickets only. Example: counter1 {A,B}, counter2 {B}; A001, A002, A003, B001 ->
    B001 = min(3, 0) = 0.
    """
    my_key = (t.created_at, t.ticket_id)
    ahead = [o.prefix for o in waiting if o is not t and (o.created_at, o.ticket_id) < my_key]
    if not listening:
        return sum(1 for p in ahead if p == t.prefix)
    return min(sum(1 for p in ahead if p in lanes) for lanes in listening)


def counter_sim_eta(t: Ticket, tickets: list[Ticket], active: list[Counter], snap: float,
                    avg_by_prefix: dict[str, float], early_call: int) -> float | None:
    """CounterSimEtaMinutes: deterministic Call Next drain (live CounterSimulationEstimator).

    Active counters only; None (= live infinity -> null) if no Active counter serves my prefix.
    Counter busy with a Called/Serving ticket -> free after max(0, avg(that service) - elapsed); idle -> 0.
    Waiting tickets in the counters' services plus me are drained: the counter that is free first takes
    the earliest (created_at, ticket_id) ticket of its services and stays busy for avg(service) minutes.
    Result = max(time my ticket is taken, minutes until my own eligibility), avg = RollingAvg rule.
    Only tickets ahead of me matter: a counter whose next ticket is after me never takes one ahead of me.
    """
    all_prefixes = set().union(*[set(c.allowed_prefixes) for c in active]) if active else set()
    if t.prefix not in all_prefixes:
        return None
    free_at: dict[int, float] = {}
    for c in active:
        busy = [o for o in tickets if o.counter_id == c.counter_id and o.called_at is not None
                and o.called_at <= snap < o.service_end_at]
        free_at[c.counter_id] = max(0.0, avg_by_prefix[busy[0].prefix] - (snap - busy[0].called_at)) if busy else 0.0
    my_key = (t.created_at, t.ticket_id)
    queue = sorted((o for o in tickets
                    if o is not t and o.prefix in all_prefixes and is_waiting(o, snap)
                    and (o.created_at, o.ticket_id) < my_key),
                   key=lambda o: (o.created_at, o.ticket_id))
    queue.append(t)
    taken: set[int] = set()
    lanes = {c.counter_id: set(c.allowed_prefixes) for c in active}
    result = None
    for _ in range(2000):
        best_c, best_ticket, best_free = None, None, float("inf")
        for c in active:
            head = next((o for o in queue if o.ticket_id not in taken and o.prefix in lanes[c.counter_id]), None)
            if head is None:
                continue
            f = free_at[c.counter_id]
            if f < best_free or (abs(f - best_free) < 1e-9 and best_ticket is not None
                                 and head.created_at < best_ticket.created_at):
                best_c, best_ticket, best_free = c.counter_id, head, f
        if best_ticket is None:
            return None
        if best_ticket is t:
            result = max(0.0, best_free)
            break
        taken.add(best_ticket.ticket_id)
        free_at[best_c] = best_free + avg_by_prefix[best_ticket.prefix]
    if result is None:
        result = 0.0
    my_eligible = t.queue_eligible_at if t.queue_eligible_at is not None else (t.slot_start or snap) - early_call
    return max(result, my_eligible - snap)


def build_counters(rng: np.random.Generator, open_min: int, close_min: int) -> list[Counter]:
    """1..8 Active counters with randomized services and availability intervals."""
    n = int(rng.choice(np.arange(1, MAX_COUNTERS + 1), p=np.array(N_COUNTER_WEIGHTS) / sum(N_COUNTER_WEIGHTS)))
    modes = list(COUNTER_MODE_WEIGHTS)
    mode_p = np.array([COUNTER_MODE_WEIGHTS[m] for m in modes])
    end_all = close_min + DRAIN_AFTER_CLOSE_MIN
    counters = []
    for i in range(n):
        mode = str(rng.choice(modes, p=mode_p / mode_p.sum()))
        if mode == "profile":
            allowed = list(PROFILES[int(rng.integers(len(PROFILES)))])
        elif mode == "random":
            k = int(rng.integers(2, len(ALL_PREFIXES)))
            allowed = [str(p) for p in rng.choice(ALL_PREFIXES, size=k, replace=False)]
        elif mode == "single":
            allowed = [str(rng.choice(ALL_PREFIXES))]
        else:
            allowed = list(ALL_PREFIXES)
        start, end = float(open_min), float(end_all)
        if i > 0 and rng.random() < P_COUNTER_LATE_OPEN:  # counter 0 opens with the branch
            start = open_min + rng.uniform(15, 180)
        if rng.random() < P_COUNTER_EARLY_CLOSE:
            end = rng.uniform(max(start + 60, open_min + 120), close_min)
        intervals = [(start, end)]
        if rng.random() < P_COUNTER_BREAK and end - start > 120:
            b0 = rng.uniform(start + 30, end - 60)
            intervals = [(start, b0), (b0 + rng.uniform(10, 60), end)]
        counters.append(Counter(counter_id=i, allowed_prefixes=allowed, intervals=intervals))
    return counters


def snapshot_row(t: Ticket, snap: float, tickets: list[Ticket], counters: list[Counter],
                 history: dict[str, list[tuple[float, float]]], ctx: dict) -> dict:
    """Features of ticket t at snap, computed exactly like live WaitTimeFeatureBuilder + BuildPayload."""
    hour = max(0, min(23, int(snap // 60)))
    active = [c for c in counters if c.is_active(snap)]
    listening, heard = heard_lanes(t.prefix, active)
    listening_counters = [c for c in active if t.prefix in c.allowed_prefixes]
    waiting = [o for o in tickets if is_waiting(o, snap)]
    in_service = [o for o in tickets if o.called_at is not None and o.called_at <= snap < o.service_end_at]
    busy_counter_ids = {o.counter_id for o in in_service}
    avg_by_prefix = {s["prefix"]: rolling_avg_service_minutes(history[s["prefix"]], snap, s["avg_min"])
                     for s in SERVICES}
    sim_eta = counter_sim_eta(t, tickets, active, snap, avg_by_prefix, ctx["early_call"])

    is_online = t.entry_type == 0
    minutes_since_slot_start = None
    minutes_until_slot_end = None
    online_booked_in_slot = 0
    walkin_in_slot = 0
    slot_active = 0
    if is_online and t.slot_start is not None:
        minutes_since_slot_start = max(0.0, snap - t.slot_start)
        minutes_until_slot_end = max(0.0, t.slot_end - snap)
        slot_active = 1 if t.slot_start <= snap < t.slot_end else 0
        # Live: same service and same AssignedSlotStart/End, any state except Missed (tickets that exist at snap)
        for o in tickets:
            if o.service_code == t.service_code and o.slot_start == t.slot_start and o.created_at <= snap:
                if o.entry_type == 0:
                    online_booked_in_slot += 1
                else:
                    walkin_in_slot += 1
    call_priority = 0 if (is_online and t.checked_in and slot_active == 1) else 1
    listening_n = len(listening_counters)
    return {
        # Queue state
        "QueueLength": sum(1 for o in waiting if o.service_code == t.service_code),
        "CrossLaneQueueLength": sum(1 for o in waiting if o.prefix in heard),
        "PeopleAheadCallNext": people_ahead_call_next(t, waiting, listening),
        "NowServing": sum(1 for o in in_service if o.service_code == t.service_code),
        # Capacity (BuildPayload: ActiveCounters >= 1; ListeningCounters = listening, else ActiveCounters, >= 1)
        "ActiveCounters": max(1, len(active)),
        "ListeningCounters": max(1, listening_n if listening_n > 0 else len(active)),
        "IdleListeningCounters": sum(1 for c in listening_counters if c.counter_id not in busy_counter_ids),
        "CounterSimEtaMinutes": None if sim_eta is None else round(sim_eta, 2),
        # Time
        "HourOfDay": hour,
        "DayOfWeek": ctx["dow"],
        "IsPeakHour": 1 if (9 <= hour <= 11 or 14 <= hour <= 16) else 0,
        # Service duration
        "DefaultAvgServiceMinutes": t.default_avg_min,
        "RollingAvgServiceMinutes": round(avg_by_prefix[t.prefix], 2),
        # Entry
        "EnqueueSequence": t.enqueue_seq,
        "EntryType": t.entry_type,
        "CheckedIn": 1 if t.checked_in else 0,
        # Slot features (walk-ins: NaN / 0 like live)
        "SlotActive": slot_active,
        "CallNextPriority": call_priority,
        "SlotDurationMinutes": SLOT_DURATION,
        "EarlyCallMinutes": ctx["early_call"],
        "MinutesUntilSlotEnd": minutes_until_slot_end,
        "MinutesSinceSlotStart": minutes_since_slot_start,
        "OnlineSlotsPerSlot": t.online_slots_per_slot,
        "OnlineBookedInSlot": online_booked_in_slot,
        "WalkInInSlot": walkin_in_slot,
        # Identity
        "BranchCode": t.branch_code,
        "ServiceCode": t.service_code,
        "TicketPrefix": t.prefix,
        # Target: CalledAt - snapshot (snapshot = QueueEligibleAt for the primary row)
        "WaitingMinutes": None if t.called_at is None else round(t.called_at - snap, 2),
    }


def simulate_day(rng: np.random.Generator, day_of_week: int) -> list[dict]:
    """Simulate one branch-day (randomized setup) with a discrete-event Call Next drain."""
    day_mult = day_traffic(day_of_week)
    if day_mult == 0:
        return []
    svc_by_prefix = {s["prefix"]: s for s in SERVICES}
    branch_code = int(rng.integers(BRANCH_CODE_MIN, BRANCH_CODE_MAX + 1))
    hours_p = np.array([p for _, p in HOURS_CHOICES])
    open_min, close_min = HOURS_CHOICES[int(rng.choice(len(HOURS_CHOICES), p=hours_p / hours_p.sum()))][0]
    end_time = float(close_min + DRAIN_AFTER_CLOSE_MIN)
    n_slots = (close_min - open_min) // SLOT_DURATION  # a slot must end by closing time
    early_call = int(rng.choice(EARLY_CALL_CHOICES))
    online_cap = {s["code"]: int(rng.integers(ONLINE_SLOTS_MIN, ONLINE_SLOTS_MAX + 1)) for s in SERVICES}
    counters = build_counters(rng, open_min, close_min)
    covered = sorted(set().union(*[set(c.allowed_prefixes) for c in counters]))

    # Branch-day service speed (true mean = default x speed) and arrival mix over the covered services
    speed = {p: float(rng.lognormal(0.0, 0.2)) for p in ALL_PREFIXES}
    w = np.array([svc_by_prefix[p]["weight"] * rng.lognormal(0.0, 0.5) for p in covered])
    w /= w.sum()
    svc_mean = float(sum(wi * svc_by_prefix[p]["avg_min"] * speed[p] for wi, p in zip(w, covered)))
    lv_p = np.array([p for _, _, p in LOAD_LEVELS])
    lo_r, hi_r, _ = LOAD_LEVELS[int(rng.choice(len(LOAD_LEVELS), p=lv_p / lv_p.sum()))]
    rho = rng.uniform(lo_r, hi_r)
    online_frac = rng.uniform(0.05, 0.6)
    hours = list(range(open_min // 60, (close_min + 59) // 60))
    h_norm = float(np.mean([hour_traffic(h) for h in hours]))

    # SERVICE_LOGS history before the day: none (fallback to default), a few (< 5 valid), or full; plus
    # seconds-long test services that the rule ignores.
    hist_modes = list(HISTORY_MODE_WEIGHTS)
    hist_p = np.array([HISTORY_MODE_WEIGHTS[m] for m in hist_modes])
    history: dict[str, list[tuple[float, float]]] = {}
    for s in SERVICES:
        mode = str(rng.choice(hist_modes, p=hist_p / hist_p.sum()))
        n_prior = 0 if mode == "none" else int(rng.integers(1, 5)) if mode == "few" else int(rng.integers(10, 41))
        true_mean = s["avg_min"] * speed[s["prefix"]]
        entries = [(-1.0, true_mean * float(rng.lognormal(-0.06, 0.35))) for _ in range(n_prior)]
        if rng.random() < 0.3:
            entries += [(-1.0, rng.uniform(2, 25) / 60.0) for _ in range(int(rng.integers(1, 6)))]
        history[s["prefix"]] = entries

    def serves_at(prefix: str, t: float) -> bool:
        return any(prefix in c.allowed_prefixes and c.is_active(t) for c in counters)

    tickets: list[Ticket] = []
    online_used: dict[tuple[str, int], int] = {}
    for hour in hours:
        lo, hi = max(hour * 60, open_min), min(hour * 60 + 60, close_min)
        if hi <= lo:
            continue
        n_open = max(0.5, sum(1 for c in counters if c.is_active((lo + hi) / 2)))
        lam = rho * hour_traffic(hour) / h_norm * day_mult * n_open * (hi - lo) / svc_mean
        for _ in range(int(rng.poisson(lam))):
            arrive = rng.uniform(lo, hi)
            prefix = str(rng.choice(covered, p=w))
            svc = svc_by_prefix[prefix]
            code = svc["code"]
            entry_type = 0 if (rng.random() < online_frac and n_slots > 0) else 1
            slot_idx = min(int((arrive - open_min) // SLOT_DURATION), n_slots - 1)
            if entry_type == 0:
                if online_used.get((code, slot_idx), 0) >= online_cap[code]:
                    entry_type = 1  # slot full -> walks in
                else:
                    online_used[(code, slot_idx)] = online_used.get((code, slot_idx), 0) + 1
            if entry_type == 0:
                slot_start = float(open_min + slot_idx * SLOT_DURATION)
                created = arrive - rng.exponential(4) * 60  # booking time, hours earlier
                checked_in = rng.random() < 0.9
                checkin_at = eligible = None
                if checked_in:
                    u = rng.random()
                    if u < 0.2:     # very early: before the early-call window opens
                        offset = -rng.uniform(20, 60)
                    elif u < 0.75:  # early
                        offset = -rng.uniform(0, 20)
                    else:           # late
                        offset = rng.uniform(0, 40)
                    checkin_at = max(open_min - 30.0, slot_start + offset)
                    eligible = max(checkin_at, slot_start - early_call)
                tickets.append(Ticket(0, branch_code, code, prefix, svc["avg_min"], online_cap[code], 0,
                                      checked_in, created, slot_start, slot_start + SLOT_DURATION,
                                      checkin_at, eligible))
            else:
                if not serves_at(prefix, arrive):
                    continue  # live WalkInAsync rejects a service with no Active counter
                bucket = float(open_min + int((arrive - open_min) // SLOT_DURATION) * SLOT_DURATION)
                tickets.append(Ticket(0, branch_code, code, prefix, svc["avg_min"], online_cap[code], 1,
                                      True, arrive, bucket, bucket + SLOT_DURATION, arrive, arrive,
                                      left=rng.random() < P_WALKIN_LEFT))
    if not tickets:
        return []
    order = rng.permutation(len(tickets))  # ticket ids = creation order is irrelevant; Id is a GUID live
    for i, t in zip(order, tickets):
        t.ticket_id = int(i) + 1
    for rank, t in enumerate(sorted(tickets, key=lambda x: (x.created_at, x.ticket_id)), start=1):
        t.enqueue_seq = rank

    # ── Discrete-event Call Next drain ──
    callable_tickets = [t for t in tickets if t.queue_eligible_at is not None]
    for c in counters:
        c.next_free = c.intervals[0][0]
    while True:
        live = [c for c in counters if not c.retired]
        if not live:
            break
        c = min(live, key=lambda x: x.next_free)
        now = c.next_free
        if now > end_time:
            c.retired = True
            continue
        if not c.is_active(now):
            later = [a for a, _ in c.intervals if a > now]
            if later:
                c.next_free = min(later)
            else:
                c.retired = True
            continue
        pool = [t for t in callable_tickets
                if t.called_at is None and t.queue_eligible_at <= now and t.prefix in c.allowed_prefixes]
        if not pool:
            future = [t.queue_eligible_at for t in callable_tickets
                      if t.called_at is None and t.queue_eligible_at > now and t.prefix in c.allowed_prefixes]
            if future:
                c.next_free = min(future)
            else:
                c.retired = True
            continue
        t = min(pool, key=lambda x: (x.created_at, x.ticket_id))  # QueueCallNextSelector
        called = now + rng.exponential(CALL_LATENCY_MEAN_MIN)
        if t.left:
            duration = rng.uniform(0.5, 3.0)  # counter waits for the absent customer, then Missed
        else:
            duration = t.default_avg_min * speed[t.prefix] * float(rng.lognormal(-0.06, 0.35))
            if rng.random() < 0.03:
                duration *= rng.uniform(2.0, 3.0)  # difficult customer
        t.called_at = called
        t.service_end_at = called + duration
        t.counter_id = c.counter_id
        c.next_free = called + duration
        if not t.left:
            history[t.prefix].append((called + duration, duration))

    # ── Snapshot rows ──
    ctx = {"dow": day_of_week, "early_call": early_call}
    rows: list[dict] = []
    for t in tickets:
        if t.called_at is None:
            if t.entry_type == 0 and not t.checked_in:  # never checked in: no target (dropped in cleaning)
                row = snapshot_row(t, t.slot_start, tickets, counters, history, ctx)
                row.update({"CheckedIn": 0, "WaitingMinutes": None})
                rows.append(row)
            continue
        rows.append(snapshot_row(t, t.queue_eligible_at, tickets, counters, history, ctx))
        start = t.checkin_at if t.checkin_at is not None else t.queue_eligible_at
        if t.called_at - start > 1.0 and rng.random() < EXTRA_SNAPSHOT_PROB:
            rows.append(snapshot_row(t, rng.uniform(start, t.called_at), tickets, counters, history, ctx))
    return rows


def generate_rows(rng: np.random.Generator, n_rows: int) -> list[dict]:
    """Simulate branch-days (Mon-Fri cycle) until n_rows rows exist."""
    all_rows: list[dict] = []
    day_count = 0
    while len(all_rows) < n_rows:
        all_rows.extend(simulate_day(rng, day_count % 5))
        day_count += 1
    return all_rows

In [ ]:
%%time
# Generate 30K rows of randomized branch-days (same as generate() in generate_seed_data.py)
rng = np.random.default_rng(42)
TARGET_ROWS = 30000
df = pd.DataFrame(generate_rows(rng, TARGET_ROWS))

# Winsorize at p99
served = df['WaitingMinutes'].notna()
if served.sum() > 0:
    p99 = df.loc[served, 'WaitingMinutes'].quantile(0.99)
    df.loc[served, 'WaitingMinutes'] = df.loc[served, 'WaitingMinutes'].clip(upper=p99)

df['DataSource'] = 'synthetic'
print(f'Generated {len(df)} rows ({served.sum()} served)')
print(f'Wait time: mean={df["WaitingMinutes"].mean():.1f}, median={df["WaitingMinutes"].median():.1f}, p95={df["WaitingMinutes"].quantile(0.95):.1f}')
print('ActiveCounters (served rows):', df.loc[served, 'ActiveCounters'].value_counts().sort_index().to_dict())

In [ ]:
# Visualize synthetic data distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

s = df[df['WaitingMinutes'].notna()]
axes[0].hist(s['WaitingMinutes'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Wait Time (min)')
axes[0].set_title(f'Wait Time Distribution (n={len(s):,})')
axes[0].axvline(s['WaitingMinutes'].median(), color='red', ls='--', label=f'median={s["WaitingMinutes"].median():.1f}')
axes[0].legend()

axes[1].bar(range(9,17), [s[s['HourOfDay']==h]['WaitingMinutes'].mean() for h in range(9,17)], color='coral')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Avg Wait (min)')
axes[1].set_title('Wait Time by Hour')

svc_avg = s.groupby('ServiceCode')['WaitingMinutes'].mean().sort_values(ascending=False)
axes[2].barh(svc_avg.index, svc_avg.values, color='mediumseagreen')
axes[2].set_xlabel('Avg Wait (min)')
axes[2].set_title('Wait Time by Service')

plt.tight_layout()
plt.show()

### 1b. Combine External Datasets (Optional)

Upload `hospital_engineered.csv` and `college_engineered.csv` to Colab, then run the cell below.
If you skip this cell, the pipeline trains on **synthetic data only** (recommended for launch).

Three safeguards prevent external data from hurting the model:
1. **Filter**: Hospital keeps only `active_counters ≥ 3` (drops 1/2-counter disaster scenarios)
2. **Scale normalization**: Align external median wait to synthetic median
3. **Sample weighting**: External gets weight 0.1 vs synthetic 1.0 during training

Note: both external datasets have a median wait of 0 min (most rows: 0 people ahead, 0 wait), so safeguard 2 never fires (scale is only applied when the source median is > 0), and their rows have no counter-state / rolling features (NaN). They are easy rows that lower the overall test MAE without improving the bank rows; compare the synthetic-only MAE.

In [ ]:
# ── Upload external datasets (Colab only) ──
# Option 1: Drag files into the left sidebar Files panel (easiest)
# Option 2: Run this cell — it will only upload files that are missing

import os

needed = ['hospital_engineered.csv', 'college_engineered.csv']
missing = [f for f in needed if not os.path.exists(f)]

if missing:
    try:
        from google.colab import files
        print(f'Missing: {missing} — uploading...')
        for f in missing:
            print(f'  Select {f}:')
            uploaded = files.upload()
    except ImportError:
        print('Not in Colab — place CSV files in working directory.')
else:
    print('Both files found (from sidebar upload). Ready for next cell.')

In [ ]:
# ── Combine External Datasets with Synthetic Data ──
# Skip this cell if you only want to train on synthetic data.

import os
TARGET = 'WaitingMinutes'
external_frames = []

# --- Hospital Queue ---
hospital_path = 'hospital_engineered.csv'  # Upload to Colab first
if os.path.exists(hospital_path):
    h = pd.read_csv(hospital_path)
    h = h.rename(columns={
        'queue_length': 'QueueLength', 'people_ahead': 'PeopleAheadCallNext',
        'now_serving': 'NowServing', 'active_counters': 'ActiveCounters',
        'hour': 'HourOfDay', 'is_peak': 'IsPeakHour',
        'priority': 'CallNextPriority', 'default_avg_service_min': 'DefaultAvgServiceMinutes',
        'wait_min': TARGET,
    })
    # Safeguard 1: Filter — drop 1/2-counter disaster scenarios
    n_before = len(h)
    h = h[h['ActiveCounters'] >= 3].copy()
    h['CrossLaneQueueLength'] = h['QueueLength']
    h['ListeningCounters'] = h['ActiveCounters']
    h['BranchCode'] = 900001
    h['ServiceCode'] = 'HOSPITAL'
    h['TicketPrefix'] = 'H'
    h['EntryType'] = 1
    h['CheckedIn'] = 1
    h['EnqueueSequence'] = range(1, len(h) + 1)
    h['DayOfWeek'] = 2
    h['DataSource'] = 'hospital'
    external_frames.append(h)
    print(f'Hospital: {len(h)} rows (filtered from {n_before}, kept counters>=3)')
else:
    print(f'Hospital file not found ({hospital_path}) — skipped')

# --- College Admissions ---
college_path = 'college_engineered.csv'  # Upload to Colab first
if os.path.exists(college_path):
    c = pd.read_csv(college_path)
    c = c.rename(columns={
        'queue_length': 'QueueLength', 'people_ahead': 'PeopleAheadCallNext',
        'now_serving': 'NowServing', 'active_counters': 'ActiveCounters',
        'hour': 'HourOfDay', 'day_of_week': 'DayOfWeek',
        'is_peak': 'IsPeakHour', 'avg_service_min': 'DefaultAvgServiceMinutes',
        'wait_min': TARGET,
    })
    c['CrossLaneQueueLength'] = c['QueueLength']
    c['ListeningCounters'] = c['ActiveCounters']
    c['BranchCode'] = 900002
    c['ServiceCode'] = 'COLLEGE'
    c['TicketPrefix'] = 'C'
    c['EntryType'] = 1
    c['CheckedIn'] = 1
    c['EnqueueSequence'] = range(1, len(c) + 1)
    c['DataSource'] = 'college'
    external_frames.append(c)
    print(f'College: {len(c)} rows')
else:
    print(f'College file not found ({college_path}) — skipped')

# --- Merge ---
if external_frames:
    ext_df = pd.concat(external_frames, ignore_index=True)

    # Safeguard 2: Scale normalization — align median to synthetic
    synthetic_median = df[TARGET].dropna().median()
    for src in ext_df['DataSource'].unique():
        mask = ext_df['DataSource'] == src
        src_median = ext_df.loc[mask, TARGET].median()
        if src_median > 0:
            scale = synthetic_median / src_median
            ext_df.loc[mask, TARGET] *= scale
            ext_df['DefaultAvgServiceMinutes'] = ext_df['DefaultAvgServiceMinutes'].astype(float)
            ext_df.loc[mask, 'DefaultAvgServiceMinutes'] *= scale
            print(f'Scaled {src}: median {src_median:.1f} -> {ext_df.loc[mask, TARGET].median():.1f} min (x{scale:.2f})')

    # Cap external at 50% of synthetic (prevent domination)
    max_ext = int(len(df) * 0.5)
    if len(ext_df) > max_ext:
        ext_df = ext_df.sample(n=max_ext, random_state=42)
        print(f'Downsampled external to {max_ext} rows (50% of synthetic)')

    df = pd.concat([df, ext_df], ignore_index=True)
    print(f'\nCombined total: {len(df)} rows')
    for src in df['DataSource'].unique():
        print(f'  {src}: {(df["DataSource"] == src).sum()}')
else:
    print('No external datasets found — training on synthetic only')

## 2. Data Cleansing

In [ ]:
TARGET = 'WaitingMinutes'
n_start = len(df)

# Drop abandoned tickets (no service = no target)
df = df.dropna(subset=[TARGET])

# Safety net: drop negative waits (system glitches). The simulation models online check-in time,
# so QueueEligibleAt <= CalledAt always holds and this removes ~nothing. Keep 0 = instant service.
df = df[df[TARGET] >= 0].copy()

# Winsorize per source
for src in df['DataSource'].unique():
    mask = df['DataSource'] == src
    p99 = df.loc[mask, TARGET].quantile(0.99)
    p01 = df.loc[mask, TARGET].quantile(0.01)
    df.loc[mask, TARGET] = df.loc[mask, TARGET].clip(lower=p01, upper=p99)

# Dedup
df = df.drop_duplicates()

print(f'Cleaning: {n_start} -> {len(df)} rows (dropped {n_start - len(df)}, {(n_start-len(df))/n_start:.1%})')
print(f'Target range: [{df[TARGET].min():.1f}, {df[TARGET].max():.1f}] min')
print(f'Mean={df[TARGET].mean():.1f}, Median={df[TARGET].median():.1f}')

## 3. Feature Engineering

**28 features** aligned with production `MlHttpWaitTimeEstimator.BuildPayload`:
- Queue state: QueueLength, CrossLaneQueueLength, PeopleAheadCallNext, NowServing
- Capacity: ActiveCounters, ListeningCounters
- Counter state: IdleListeningCounters, CounterSimEtaMinutes (NaN when no Active counter serves the lane, and for external datasets)
- Time: HourOfDay, DayOfWeek, IsPeakHour
- Service: DefaultAvgServiceMinutes, RollingAvgServiceMinutes (live rule; falls back to the default)
- Entry: EnqueueSequence, EntryType, CheckedIn
- Slot (NaN for walk-ins): SlotActive, CallNextPriority, SlotDurationMinutes, MinutesUntilSlotEnd, MinutesSinceSlotStart, OnlineSlotsPerSlot, OnlineBookedInSlot, WalkInInSlot
- Branch policy: EarlyCallMinutes (= Branch.OnlineEarlyCallMinutes, set by manager 0..30; NaN for external datasets)
- Identity: BranchCode, ServiceCode, TicketPrefix

In [ ]:
FEATURE_NUMERIC = [
    'QueueLength', 'CrossLaneQueueLength', 'PeopleAheadCallNext', 'NowServing',
    'ActiveCounters', 'ListeningCounters',
    'IdleListeningCounters', 'CounterSimEtaMinutes',
    'HourOfDay', 'DayOfWeek', 'IsPeakHour',
    'DefaultAvgServiceMinutes',
    'RollingAvgServiceMinutes',
    'EnqueueSequence', 'EntryType', 'CheckedIn',
    'SlotActive', 'CallNextPriority', 'SlotDurationMinutes', 'EarlyCallMinutes',
    'MinutesUntilSlotEnd', 'MinutesSinceSlotStart',
    'OnlineSlotsPerSlot', 'OnlineBookedInSlot', 'WalkInInSlot',
    'BranchCode',
]
FEATURE_CAT = ['ServiceCode', 'TicketPrefix']
ALL_FEATURES = FEATURE_NUMERIC + FEATURE_CAT

# Ensure columns exist
for col in FEATURE_NUMERIC:
    if col not in df.columns: df[col] = np.nan
    df[col] = pd.to_numeric(df[col], errors='coerce')
for col in FEATURE_CAT:
    if col not in df.columns: df[col] = 'UNKNOWN'

# Report NaN (expected for slot features on walk-ins, RollingAvg on external data)
nan_counts = df[FEATURE_NUMERIC].isna().sum()
nan_feats = nan_counts[nan_counts > 0]
print(f'{len(ALL_FEATURES)} features ({len(FEATURE_NUMERIC)} numeric + {len(FEATURE_CAT)} categorical)')
if len(nan_feats) > 0:
    print('\nNaN features (handled natively by HistGBR):')
    for f, c in nan_feats.items():
        print(f'  {f}: {c} ({c/len(df):.1%})')

## 4. Model Training

- **HistGradientBoostingRegressor** with **quantile loss (p75)** — asymmetric: under-prediction hurts UX more than over-prediction. Predicting p75 means 75% of customers are served within the estimated time.
- **Temporal split** (train on earlier data, test on later) to prevent data leakage
- **No imputation needed** — HistGBR handles NaN natively for slot features
- **Monotonic constraints**: increasing in PeopleAheadCallNext, CrossLaneQueueLength, CounterSimEtaMinutes; decreasing in IdleListeningCounters, ListeningCounters

In [ ]:
# Per-source temporal split (same as train_wait_model.py)
# Each DataSource is split independently to prevent all external data
# ending up on one side (this bug caused MAE=468 before we fixed it).

sort_col = 'EnqueueSequence'
test_ratio = 0.2

if 'DataSource' not in df.columns:
    df['DataSource'] = 'synthetic'

train_parts, test_parts = [], []
for source, group in df.groupby('DataSource'):
    group = group.sort_values(sort_col)
    split_idx = int(len(group) * (1 - test_ratio))
    train_parts.append(group.iloc[:split_idx])
    test_parts.append(group.iloc[split_idx:])

train_df = pd.concat(train_parts, ignore_index=True)
test_df  = pd.concat(test_parts, ignore_index=True)

X_train, y_train = train_df[ALL_FEATURES], train_df[TARGET].astype(float)
X_test,  y_test  = test_df[ALL_FEATURES],  test_df[TARGET].astype(float)

print(f'Per-source temporal split: train={len(train_df)}, test={len(test_df)} ({test_ratio:.0%} test)')
for src in df['DataSource'].unique():
    n_tr = (train_df['DataSource'] == src).sum()
    n_te = (test_df['DataSource'] == src).sum()
    print(f'  {src}: train={n_tr}, test={n_te}')
print(f'Train wait: mean={y_train.mean():.1f}, Test wait: mean={y_test.mean():.1f}')

In [ ]:
%%time
# Build and train model
# Monotonic constraints (same as train_wait_model.py): more people ahead / a longer simulated drain / more waiting
# in the heard lanes never shortens the ETA; more listening / idle counters never lengthens it.
MONOTONIC = {'PeopleAheadCallNext': 1, 'CrossLaneQueueLength': 1, 'CounterSimEtaMinutes': 1,
             'IdleListeningCounters': -1, 'ListeningCounters': -1}
preprocessor = ColumnTransformer([
    ('num', 'passthrough', FEATURE_NUMERIC),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), FEATURE_CAT),
])

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', HistGradientBoostingRegressor(
        loss='quantile',       # asymmetric: under-prediction hurts UX more than over-prediction
        quantile=0.75,         # predict p75 — 75% of customers served within this estimate
        max_depth=6, learning_rate=0.05, max_iter=600,
        min_samples_leaf=8, random_state=42,
        # column order after the ColumnTransformer = FEATURE_NUMERIC + FEATURE_CAT
        monotonic_cst=[MONOTONIC.get(f, 0) for f in FEATURE_NUMERIC] + [0] * len(FEATURE_CAT),
    )),
])

# Safeguard 3: Sample weighting — synthetic/primary=1.0, external=0.1
# This ensures external data adds signal without dominating the loss function.
train_weights = train_df['DataSource'].map(
    lambda s: 1.0 if s in ('synthetic', 'primary', 'sql') else 0.1
).values
pipeline.fit(X_train, y_train, model__sample_weight=train_weights)

weight_summary = train_df['DataSource'].value_counts()
print('Model trained with quantile loss (p75) and sample weighting:')
print('  p75 = 75% of customers served within the predicted time (conservative estimate)')
for src in weight_summary.index:
    w = 1.0 if src in ('synthetic', 'primary', 'sql') else 0.1
    print(f'  {src}: {weight_summary[src]} rows x weight {w}')

## 5. Evaluation

In [ ]:
# ML predictions
ml_pred = np.maximum(0, pipeline.predict(X_test))

# Formula baseline (matches production WaitTimeEstimator.cs)
def formula_baseline(row):
    ahead = max(0, row.get('PeopleAheadCallNext', 0) or 0)
    now_s = max(0, row.get('NowServing', 0) or 0)
    avg = row.get('DefaultAvgServiceMinutes', 10) or 10
    counters = max(1, row.get('ListeningCounters', 1) or 1)
    return max(0, (ahead + now_s) * avg / counters)

fm_pred = test_df.apply(formula_baseline, axis=1).values

# Metrics
ml_mae  = mean_absolute_error(y_test, ml_pred)
ml_rmse = np.sqrt(mean_squared_error(y_test, ml_pred))
ml_r2   = r2_score(y_test, ml_pred)
fm_mae  = mean_absolute_error(y_test, fm_pred)
fm_rmse = np.sqrt(mean_squared_error(y_test, fm_pred))
fm_r2   = r2_score(y_test, fm_pred)

# Cross-validation (on full dataset)
# FIX: Sort by DataSource → BranchCode → EnqueueSequence so TimeSeriesSplit
# respects temporal order within each branch. Previously sorted only by
# EnqueueSequence, which is independently numbered per branch/source —
# mixing them destroyed the physical time ordering.
cv_sort_cols = ['DataSource', 'BranchCode', 'EnqueueSequence']
X_all = df.sort_values(cv_sort_cols)[ALL_FEATURES]
y_all = df.sort_values(cv_sort_cols)[TARGET].astype(float)
tscv = TimeSeriesSplit(n_splits=5)
cv_mae = -cross_val_score(pipeline, X_all, y_all, cv=tscv, scoring='neg_mean_absolute_error')

print('=' * 60)
print('  RESULTS')
print('=' * 60)
print(f'  ML Model:     MAE = {ml_mae:.2f} min   RMSE = {ml_rmse:.2f}   R2 = {ml_r2:.4f}')
print(f'  Formula:      MAE = {fm_mae:.2f} min   RMSE = {fm_rmse:.2f}   R2 = {fm_r2:.4f}')
print(f'  CV MAE:       {np.mean(cv_mae):.2f} min (5-fold TimeSeriesSplit)')
print(f'  Improvement:  ML is {fm_mae/ml_mae:.1f}x more accurate than formula')

In [ ]:
# Feature Importance
from sklearn.inspection import permutation_importance

perm = permutation_importance(pipeline, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
imp = sorted(zip(ALL_FEATURES, perm.importances_mean), key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(10, 6))
names, vals = zip(*imp[:15])
ax.barh(range(len(names)), vals, color='steelblue')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_xlabel('Permutation Importance')
ax.set_title('Top 15 Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual + Residuals
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Predicted vs Actual
ax = axes[0]
ax.scatter(y_test, ml_pred, s=4, alpha=0.3, color='steelblue', label='ML')
ax.scatter(y_test, fm_pred, s=4, alpha=0.15, color='orange', label='Formula')
lim = [0, max(y_test.max(), ml_pred.max()) * 1.05]
ax.plot(lim, lim, 'r--', lw=1, label='Perfect')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('Actual (min)'); ax.set_ylabel('Predicted (min)')
ax.set_title('Predicted vs Actual')
ax.legend(markerscale=4)

# Residual histogram
ax2 = axes[1]
ml_resid = ml_pred - y_test.values
fm_resid = fm_pred - y_test.values
rng_val = max(abs(ml_resid).max(), 30)
ax2.hist(ml_resid, bins=50, range=(-rng_val, rng_val), alpha=0.6, color='steelblue', label=f'ML (MAE={ml_mae:.1f})')
ax2.hist(fm_resid, bins=50, range=(-rng_val, rng_val), alpha=0.4, color='orange', label=f'Formula (MAE={fm_mae:.1f})')
ax2.axvline(0, color='red', ls='--')
ax2.set_xlabel('Residual (min)'); ax2.set_title('Residual Distribution')
ax2.legend()

# Error by service type
ax3 = axes[2]
test_df_copy = test_df.copy()
test_df_copy['ml_error'] = np.abs(ml_pred - y_test.values)
test_df_copy['fm_error'] = np.abs(fm_pred - y_test.values)
svc_ml = test_df_copy.groupby('ServiceCode')['ml_error'].mean().sort_values(ascending=False)
svc_fm = test_df_copy.groupby('ServiceCode')['fm_error'].mean()
x_pos = range(len(svc_ml))
ax3.bar([p - 0.2 for p in x_pos], svc_ml.values, 0.4, color='steelblue', label='ML')
ax3.bar([p + 0.2 for p in x_pos], [svc_fm.get(s, 0) for s in svc_ml.index], 0.4, color='orange', label='Formula')
ax3.set_xticks(list(x_pos))
ax3.set_xticklabels(svc_ml.index, rotation=45, ha='right', fontsize=8)
ax3.set_ylabel('MAE (min)'); ax3.set_title('MAE by Service Type')
ax3.legend()

plt.suptitle('QMS Wait-Time Model Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

| Metric | ML Model | Formula Baseline |
|--------|----------|------------------|
| MAE    | ~13-14 min | ~28 min |
| R²     | ~0.82-0.83 | < 0 (worse than mean) |
| Improvement | **~2x** more accurate | baseline |

(Synthetic-only, temporal split on randomized branch-days incl. 1-2 counter days and over-capacity days.)

### Architecture
```
Customer takes ticket
    → WaitTimeFeatureBuilder computes 28 features
    → MlHttpWaitTimeEstimator sends to Python sidecar
    → HistGradientBoosting predicts wait time
    → Display the raw ML value (no clamp)
    → Display on ticket / app
```

Synthetic data models online check-in time, so online tickets only become eligible after the customer checks in (no negative waits).

### Deployment Phases
1. **Pre-launch**: Train on synthetic data (this notebook)
2. **Post-launch**: Retrain on real `ML_TRAINING_DATA` from system
3. **Mature**: Full real data, consider Two-Stage model for anomalies

In [ ]:
import joblib, json
from pathlib import Path

MODEL_PATH = Path('wait_model.joblib')
METRICS_PATH = Path('model_metrics.json')

# ── Export model bundle (same format as train_wait_model.py) ──
joblib.dump({
    'pipeline': pipeline,
    'feature_numeric': FEATURE_NUMERIC,
    'feature_categorical': FEATURE_CAT,
    'target': TARGET,
    'version': 'v4',
}, MODEL_PATH)

# ── Export metrics ──
metrics = {
    'data_source': 'notebook (synthetic)',
    'row_count': len(df),
    'test_ml': {'mae_minutes': round(ml_mae, 3), 'rmse_minutes': round(ml_rmse, 3), 'r2': round(ml_r2, 4)},
    'test_formula_baseline': {'mae_minutes': round(fm_mae, 3), 'rmse_minutes': round(fm_rmse, 3), 'r2': round(fm_r2, 4)},
    'cv_mae_mean': round(float(np.mean(cv_mae)), 3),
    'feature_count': len(ALL_FEATURES),
    'features_numeric': FEATURE_NUMERIC,
    'features_categorical': FEATURE_CAT,
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2))

print(f'Saved: {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1024:.0f} KB)')
print(f'Saved: {METRICS_PATH}')
print(f'\nDeploy: copy wait_model.joblib to ml/ then restart serve_wait_model.py')